In [ ]:
%%bash
curl -k "https://orthanc.katelyncmorrison.com/studies/8022c382-9a178579-70170824-d040fc0e-12f6f132/series?expand" \
  | jq '.[]
       | { SeriesID: .ID,
           SeriesNumber: .MainDicomTags.SeriesNumber,
           SeriesDescription: .MainDicomTags.SeriesDescription,
           Modality: .MainDicomTags.Modality }'


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  120k  100  120k    0     0   281k      0 --:--:-- --:--:-- --:--:--  280k


{
  "SeriesID": "c8903fa5-bbca061d-8c5f69e7-17a98c10-64355063",
  "SeriesNumber": "1",
  "SeriesDescription": "Generation 4",
  "Modality": "AI"
}
{
  "SeriesID": "314bfcdf-16a2c020-a9187727-a37aff2c-185c943c",
  "SeriesNumber": "1",
  "SeriesDescription": "Bilateral Moderate Pleural Effusion",
  "Modality": "AI"
}
{
  "SeriesID": "fb9081ba-3f35144c-99df0ec7-ab76349a-0d2d2a34",
  "SeriesNumber": "1",
  "SeriesDescription": "Normal chest with no abnormalities present",
  "Modality": "AI"
}
{
  "SeriesID": "63581e67-db2f8414-0eae7bb4-ae82cc4d-51026924",
  "SeriesNumber": "1",
  "SeriesDescription": "Bilateral Large Pleural Effusion",
  "Modality": "AI"
}
{
  "SeriesID": "1054e5b0-5051f740-4012daff-5af43f67-f3462bc2",
  "SeriesNumber": "1",
  "SeriesDescription": "Bilateral minimal pleural effusion with no other associated findings, no cardiomegaly, no ground glass, no atelectasis, no nodules, no consolidation",
  "Modality": "AI"
}
{
  "SeriesID": "9b8369cc-66f98785-1a4e9b3d-65936dde-c37

In [ ]:
"Bilateral Moderate Pleural Effusion": "Bilateral moderate pleural effusion with no other associated findings, no cardiomegaly, no ground glass, no atelectasis, no nodules, no consolidation",
"Minimal Left": "Minimal pleural effusion in the left hemithorax",
"Bilateral minimal pleural effusion with no other associated findings, no cardiomegaly, no ground glass, no atelectasis, no nodules, no consolidation": "Bilateral minimal pleural effusion with no other associated findings, no cardiomegaly, no ground glass, no atelectasis, no nodules, no consolidation",
"Bilateral Large Pleural Effusion": "Bilateral large pleural effusion with no other associated findings, no cardiomegaly, no ground glass, no atelectasis, no nodules, no consolidation",
"Bilateral Moderate 2": "Bilateral moderate pleural effusion with no other associated findings, no cardiomegaly, no ground glass, no atelectasis, no nodules, no consolidation",
"Bilateral Moderate 3": "Moderate pleural effusion in the left and right lungs",
"Generation 4": "Severe right pleural effusion associated with passive atelectasis of the right lower lobe",
"Normal chest with no abnormalities present": "Normal chest with no abnormalities present"


In [3]:
import requests
import urllib3

# ---- CONFIGURATION ----
ORTHANC_BASE = "https://orthanc.katelyncmorrison.com"
STUDY_ID     = "8022c382-9a178579-70170824-d040fc0e-12f6f132"
# your mapping of SeriesDescription -> prompt
PROMPT_MAP = {
    "Bilateral Moderate Pleural Effusion":
      "Bilateral moderate pleural effusion with no other associated findings, no cardiomegaly, no ground glass, no atelectasis, no nodules, no consolidation",
    "Minimal Left":
      "Minimal pleural effusion in the left hemithorax",
    "Bilateral minimal pleural effusion with no other associated findings, no cardiomegaly, no ground glass, no atelectasis, no nodules, no consolidation":
      "Bilateral minimal pleural effusion with no other associated findings, no cardiomegaly, no ground glass, no atelectasis, no nodules, no consolidation",
    "Bilateral Large Pleural Effusion":
      "Bilateral large pleural effusion with no other associated findings, no cardiomegaly, no ground glass, no atelectasis, no nodules, no consolidation",
    "Bilateral Moderate 2":
      "Bilateral moderate pleural effusion with no other associated findings, no cardiomegaly, no ground glass, no atelectasis, no nodules, no consolidation",
    "Bilateral Moderate 3":
      "Moderate pleural effusion in the left and right lungs",
    "Generation 4":
      "Severe right pleural effusion associated with passive atelectasis of the right lower lobe",
    "Normal chest with no abnormalities present":
      "Normal chest with no abnormalities present",
}

# disable SSL warnings for -k
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

def main():
    # 1) fetch the expanded series list
    url = f"{ORTHANC_BASE}/studies/{STUDY_ID}/series?expand"
    resp = requests.get(url, verify=False)
    resp.raise_for_status()
    series_list = resp.json()

    for series in series_list:
        sid  = series["ID"]
        desc = series.get("MainDicomTags",{}).get("SeriesDescription")
        prompt = PROMPT_MAP.get(desc)
        if not prompt:
            print(f"⚠️  no prompt defined for description: {desc!r}  (series {sid})")
            continue

        # 2) PUT the prompt
        put_url = f"{ORTHANC_BASE}/series/{sid}/metadata/SeriesPrompt"
        p = requests.put(put_url,
                         data=prompt.encode("utf-8"),
                         headers={"Content-Type":"application/json"},
                         verify=False)
        if p.ok:
            print(f"✅  set SeriesPrompt on {sid!r} → {desc!r}")
        else:
            print(f"❌  failed for {sid!r}: {p.status_code} {p.text}")

if __name__=="__main__":
    main()


✅  set SeriesPrompt on 'c8903fa5-bbca061d-8c5f69e7-17a98c10-64355063' → 'Generation 4'
✅  set SeriesPrompt on '314bfcdf-16a2c020-a9187727-a37aff2c-185c943c' → 'Bilateral Moderate Pleural Effusion'
✅  set SeriesPrompt on 'fb9081ba-3f35144c-99df0ec7-ab76349a-0d2d2a34' → 'Normal chest with no abnormalities present'
✅  set SeriesPrompt on '63581e67-db2f8414-0eae7bb4-ae82cc4d-51026924' → 'Bilateral Large Pleural Effusion'
✅  set SeriesPrompt on '1054e5b0-5051f740-4012daff-5af43f67-f3462bc2' → 'Bilateral minimal pleural effusion with no other associated findings, no cardiomegaly, no ground glass, no atelectasis, no nodules, no consolidation'
✅  set SeriesPrompt on '9b8369cc-66f98785-1a4e9b3d-65936dde-c37b6230' → 'Bilateral Moderate 2'
✅  set SeriesPrompt on 'aa77768b-d6e100cb-5caf54c1-4fb657b3-cd476113' → 'Bilateral Moderate 3'
✅  set SeriesPrompt on '3ec1c3b2-a32795bb-33692164-a4fa45b8-fc6b9793' → 'Minimal Left'


In [7]:
import requests

def add_metadata_to_series(series_id, data, type):
    if not (type == 'SeriesPromptBoolean'):
        print(f"Invalid metadata type: {type}.")
        return
    print(series_id)
    try:
        url = f'https://orthanc.katelyncmorrison.com/series/{series_id}/metadata/{type}'
        headers = {
            'Content-Type': 'text/plain'
        }

        response = requests.put(url, headers=headers, data=data, verify=False)
        if response.status_code != 200:
            print(f"Response not ok. Status: {response.status_code}, Response text: {response.text}")
    except requests.exceptions.RequestException as e:
        print(f'There was a problem with your fetch operation: {e}')


add_metadata_to_series("c8903fa5-bbca061d-8c5f69e7-17a98c10-64355063", "Testing metadata", "SeriesPromptBoolean")

c8903fa5-bbca061d-8c5f69e7-17a98c10-64355063
Response not ok. Status: 404, Response text: {
   "HttpError" : "Not Found",
   "HttpStatus" : 404,
   "Message" : "Accessing an inexistent item",
   "Method" : "PUT",
   "OrthancError" : "Accessing an inexistent item",
   "OrthancStatus" : 7,
   "Uri" : "/series/c8903fa5-bbca061d-8c5f69e7-17a98c10-64355063/metadata/SeriesPromptBoolean"
}



In [11]:
%%bash
curl -k -X PUT https://orthanc.katelyncmorrison.com/series/c8903fa5-bbca061d-8c5f69e7-17a98c10-64355063/metadata/SeriesPromptChanged \
     -H "Content-Type: text/plain" \
     --data "true"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100     4    0     0  100     4      0     17 --:--:-- --:--:-- --:--:--    17


In [16]:
%%bash
curl -k https://orthanc.katelyncmorrison.com/series/c8903fa5-bbca061d-8c5f69e7-17a98c10-64355063/metadata


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   125  100   125    0     0    704      0 --:--:-- --:--:-- --:--:--   706


[
   "RemoteAET",
   "LastUpdate",
   "MainDicomTagsSignature",
   "SeriesPrompt",
   "Feedback",
   "SeriesPromptChanged"
]
